In [1]:
# ======================================================================
# 1. DATA
# Hourly close prices and dollar volume for 9 large coins on Binance.US,
# Jan 2020 - Aug 2026.
#   px  : close price (last trade in each hour), hours x coins
#   vol : quote_volume = USDT traded in each hour, hours x coins
#   ret : simple hourly return, px.pct_change()
# Both tables are reindexed onto a complete hourly calendar so that one
# row = exactly one hour (missing hours become NaN rows instead of being
# silently skipped by shift / pct_change).
# Rows are labeled by each bar's OPEN time but hold that bar's CLOSE price.
# ======================================================================
from binance.client import Client as bnb_client
from datetime import datetime
import pandas as pd
import numpy as np
import statsmodels.api as sm
pd.set_option('display.float_format', '{:,.4f}'.format)

client = bnb_client(tld='US')   # drop tld='US' if you're outside the US

# Download one coin's hourly bars (klines) and convert ms timestamps to datetimes.
def get_binance_px(symbol, freq, start_ts='2020-01-01', end_ts='2026-8-31'):
    data = client.get_historical_klines(symbol, freq, start_ts, end_ts)
    columns = ['open_time','open','high','low','close','volume','close_time',
               'quote_volume','num_trades','taker_base_volume','taker_quote_volume','ignore']
    data = pd.DataFrame(data, columns=columns)
    data['open_time'] = data['open_time'].map(lambda x: datetime.utcfromtimestamp(x/1000))
    data['close_time'] = data['close_time'].map(lambda x: datetime.utcfromtimestamp(x/1000))
    return data

# Universe: 9 liquid coins quoted in USDT.
univ = ['BTCUSDT','ETHUSDT','ADAUSDT','BNBUSDT',
        'LTCUSDT','BCHUSDT','XLMUSDT','ETCUSDT','DOGEUSDT']

freq = '1h'
# One download per coin; keep both the close price and the dollar volume.
px = {}
vol = {}
for x in univ:
    data = get_binance_px(x, freq).set_index('open_time')
    px[x] = data['close']
    vol[x] = data['quote_volume']
# Binance returns numbers as text -> convert to float, then force a full hourly calendar.
px = pd.DataFrame(px).astype(float)
px = px.reindex(pd.date_range(px.index[0], px.index[-1], freq=freq))
vol = pd.DataFrame(vol).astype(float)
vol = vol.reindex(pd.date_range(vol.index[0], vol.index[-1], freq=freq))
# Hourly return of each coin (row t = return from the close of t-1 to the close of t).
ret = px.pct_change()

C:\Users\sahaj\AppData\Local\Temp\ipykernel_6960\846228514.py:15: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  data['open_time'] = data['open_time'].map(lambda x: datetime.utcfromtimestamp(x/1000))
C:\Users\sahaj\AppData\Local\Temp\ipykernel_6960\846228514.py:16: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  data['close_time'] = data['close_time'].map(lambda x: datetime.utcfromtimestamp(x/1000))
C:\Users\sahaj\AppData\Local\Temp\ipykernel_6960\846228514.py:15: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to repres

In [2]:
# ======================================================================
# 2. STEP-BY-STEP WALKTHROUGH OF THE STRATEGY (illustration only)
# The next ten cells build the strategy one step at a time so each
# intermediate table can be inspected. The real pipeline is run_strategy()
# in section 3; these cells use hold = 5 purely as an example.
# ======================================================================
# Signal: short-horizon REVERSAL. Average return over the last `hor` hours,
# flipped in sign, so the coins that fell the most get the highest signal.
hor=1
signal = ret.rolling(hor, min_periods=1).mean()*-1
signal

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,-0.0051,-0.0109,-0.0061,-0.0062,-0.0087,-0.0153,-0.0071,-0.0110,-0.0050
2020-01-01 02:00:00,-0.0036,-0.0047,-0.0061,-0.0030,-0.0024,-0.0016,-0.0013,-0.0069,-0.0009
2020-01-01 03:00:00,0.0010,0.0056,0.0039,0.0007,0.0022,0.0075,0.0015,0.0072,0.0042
2020-01-01 04:00:00,0.0020,-0.0008,0.0039,0.0017,-0.0051,-0.0046,-0.0009,-0.0008,-0.0023
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,0.0032,0.0043,0.0083,0.0021,0.0058,0.0087,0.0089,0.0160,0.0053
2026-08-30 21:00:00,-0.0003,0.0038,0.0070,0.0022,0.0008,-0.0016,0.0123,0.0054,0.0073
2026-08-30 22:00:00,0.0025,0.0043,0.0108,0.0041,0.0010,0.0056,-0.0054,-0.0014,0.0119
2026-08-30 23:00:00,0.0095,0.0216,0.0255,0.0120,0.0367,0.0365,0.0229,0.0177,0.0164


In [3]:
# Rank the signal across coins within each hour (1 = lowest signal).
rank = signal.rank(axis=1)
rank

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,8.0000,3.0000,7.0000,6.0000,4.0000,1.0000,5.0000,2.0000,9.0000
2020-01-01 02:00:00,4.0000,3.0000,2.0000,5.0000,6.0000,7.0000,8.0000,1.0000,9.0000
2020-01-01 03:00:00,2.0000,7.0000,5.0000,1.0000,4.0000,9.0000,3.0000,8.0000,6.0000
2020-01-01 04:00:00,8.0000,5.0000,9.0000,7.0000,1.0000,2.0000,4.0000,6.0000,3.0000
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,2.0000,3.0000,6.0000,1.0000,5.0000,7.0000,8.0000,9.0000,4.0000
2026-08-30 21:00:00,2.0000,5.0000,7.0000,4.0000,3.0000,1.0000,9.0000,6.0000,8.0000
2026-08-30 22:00:00,4.0000,6.0000,8.0000,5.0000,3.0000,7.0000,1.0000,2.0000,9.0000
2026-08-30 23:00:00,1.0000,5.0000,7.0000,2.0000,9.0000,8.0000,6.0000,4.0000,3.0000


In [4]:
# Average rank in each hour (the middle of the cross-section).
average_rank = rank.mean(axis=1)
average_rank

2020-01-01 00:00:00      NaN
2020-01-01 01:00:00   5.0000
2020-01-01 02:00:00   5.0000
2020-01-01 03:00:00   5.0000
2020-01-01 04:00:00   5.0000
                       ...  
2026-08-30 20:00:00   5.0000
2026-08-30 21:00:00   5.0000
2026-08-30 22:00:00   5.0000
2026-08-30 23:00:00   5.0000
2026-08-31 00:00:00   5.0000
Freq: h, Length: 58417, dtype: float64

In [5]:
# Demean: rank minus the hour's average rank. Positive = long, negative = short.
# Longs and shorts cancel, so the portfolio is dollar-neutral.
deviance = rank.subtract(average_rank, axis=0)
deviance

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,3.0000,-2.0000,2.0000,1.0000,-1.0000,-4.0000,0.0000,-3.0000,4.0000
2020-01-01 02:00:00,-1.0000,-2.0000,-3.0000,0.0000,1.0000,2.0000,3.0000,-4.0000,4.0000
2020-01-01 03:00:00,-3.0000,2.0000,0.0000,-4.0000,-1.0000,4.0000,-2.0000,3.0000,1.0000
2020-01-01 04:00:00,3.0000,0.0000,4.0000,2.0000,-4.0000,-3.0000,-1.0000,1.0000,-2.0000
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,-3.0000,-2.0000,1.0000,-4.0000,0.0000,2.0000,3.0000,4.0000,-1.0000
2026-08-30 21:00:00,-3.0000,0.0000,2.0000,-1.0000,-2.0000,-4.0000,4.0000,1.0000,3.0000
2026-08-30 22:00:00,-1.0000,1.0000,3.0000,0.0000,-2.0000,2.0000,-4.0000,-3.0000,4.0000
2026-08-30 23:00:00,-4.0000,0.0000,2.0000,-3.0000,4.0000,3.0000,1.0000,-1.0000,-2.0000


In [6]:
# Normalize so the absolute weights sum to 1 each hour
# (gross exposure 100%: 50% long, 50% short).
total = deviance.abs().sum(axis=1)
weight = deviance.div(total,axis=0)
weight

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,0.1500,-0.1000,0.1000,0.0500,-0.0500,-0.2000,0.0000,-0.1500,0.2000
2020-01-01 02:00:00,-0.0500,-0.1000,-0.1500,0.0000,0.0500,0.1000,0.1500,-0.2000,0.2000
2020-01-01 03:00:00,-0.1500,0.1000,0.0000,-0.2000,-0.0500,0.2000,-0.1000,0.1500,0.0500
2020-01-01 04:00:00,0.1500,0.0000,0.2000,0.1000,-0.2000,-0.1500,-0.0500,0.0500,-0.1000
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,-0.1500,-0.1000,0.0500,-0.2000,0.0000,0.1000,0.1500,0.2000,-0.0500
2026-08-30 21:00:00,-0.1500,0.0000,0.1000,-0.0500,-0.1000,-0.2000,0.2000,0.0500,0.1500
2026-08-30 22:00:00,-0.0500,0.0500,0.1500,0.0000,-0.1000,0.1000,-0.2000,-0.1500,0.2000
2026-08-30 23:00:00,-0.2000,0.0000,0.1000,-0.1500,0.2000,0.1500,0.0500,-0.0500,-0.1000


In [7]:
# Rebalance only every `hold` hours: keep weights on rebalance hours, blank
# the rest, then carry the last weights forward (ffill).
hold = 5

rebal = pd.Series(np.arange(len(weight)) % hold == 0, index = weight.index)
weight = weight.where(rebal, axis=0).ffill()
weight

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,0.1500,0.0500,-0.2000,0.0000,0.1000,-0.0500,-0.1000,0.2000,-0.1500
2026-08-30 21:00:00,0.1500,0.0500,-0.2000,0.0000,0.1000,-0.0500,-0.1000,0.2000,-0.1500
2026-08-30 22:00:00,0.1500,0.0500,-0.2000,0.0000,0.1000,-0.0500,-0.1000,0.2000,-0.1500
2026-08-30 23:00:00,-0.2000,0.0000,0.1000,-0.1500,0.2000,0.1500,0.0500,-0.0500,-0.1000


In [8]:
# Strategy return per coin = weight held during the hour x that hour's return.
#Shift by a factor of 1 is required -  adjust for the mismatch between open time and close price (this is essentially removing lookahead bias)
strat_ret = ret * weight.shift(1)
strat_ret

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,-0.0005,-0.0002,0.0017,-0.0000,-0.0006,0.0004,0.0009,-0.0032,0.0008
2026-08-30 21:00:00,0.0001,-0.0002,0.0014,-0.0000,-0.0001,-0.0001,0.0012,-0.0011,0.0011
2026-08-30 22:00:00,-0.0004,-0.0002,0.0022,-0.0000,-0.0001,0.0003,-0.0005,0.0003,0.0018
2026-08-30 23:00:00,-0.0014,-0.0011,0.0051,-0.0000,-0.0037,0.0018,0.0023,-0.0035,0.0025


In [9]:
# Portfolio (gross) return each hour = sum across coins.
port_ret = strat_ret.sum(axis=1)
port_ret

2020-01-01 00:00:00    0.0000
2020-01-01 01:00:00    0.0000
2020-01-01 02:00:00    0.0000
2020-01-01 03:00:00    0.0000
2020-01-01 04:00:00    0.0000
                        ...  
2026-08-30 20:00:00   -0.0007
2026-08-30 21:00:00    0.0023
2026-08-30 22:00:00    0.0033
2026-08-30 23:00:00    0.0019
2026-08-31 00:00:00    0.0050
Freq: h, Length: 58417, dtype: float64

In [10]:
# Transaction costs. Turnover = total absolute change in weights between
# consecutive hours, lagged the same way as the returns.
# Cost per hour = turnover x cost per unit traded (20 bps here).
bps = 20/10000
turnover = weight.shift(1).subtract(weight.shift(2)).abs().sum(axis=1)
tx_cost = turnover * bps
tx_cost

2020-01-01 00:00:00   0.0000
2020-01-01 01:00:00   0.0000
2020-01-01 02:00:00   0.0000
2020-01-01 03:00:00   0.0000
2020-01-01 04:00:00   0.0000
                       ...  
2026-08-30 20:00:00   0.0000
2026-08-30 21:00:00   0.0000
2026-08-30 22:00:00   0.0000
2026-08-30 23:00:00   0.0000
2026-08-31 00:00:00   0.0032
Freq: h, Length: 58417, dtype: float64

In [11]:
# Net return = gross return minus transaction costs.
net_ret = port_ret - tx_cost
net_ret

2020-01-01 00:00:00    0.0000
2020-01-01 01:00:00    0.0000
2020-01-01 02:00:00    0.0000
2020-01-01 03:00:00    0.0000
2020-01-01 04:00:00    0.0000
                        ...  
2026-08-30 20:00:00   -0.0007
2026-08-30 21:00:00    0.0023
2026-08-30 22:00:00    0.0033
2026-08-30 23:00:00    0.0019
2026-08-31 00:00:00    0.0018
Freq: h, Length: 58417, dtype: float64

In [12]:
# ======================================================================
# 3. BACKTEST FUNCTION
# run_strategy(): the full pipeline from the walkthrough in one function.
# Inputs:
#   ret       : hourly returns (hours x coins)
#   hor       : lookback in hours for the reversal signal
#   hold      : rebalance every `hold` hours
#   bps       : cost per unit of turnover (20/10000 = 20 bps)
#   tradeable : optional True/False mask (hours x coins). None = trade all
#               coins (the original strategy). If given, coins that are not
#               tradeable in an hour get no signal and zero weight.
# Returns an hourly DataFrame: gross return, net return, turnover.
# Regression check: with no mask it reproduces the original results exactly.
# ======================================================================
def run_strategy(ret, hor=1, hold=1, bps=20/10000, tradeable = None):

    # signal
    signal = ret.rolling(hor, min_periods=1).mean()*-1
    # Liquidity filter (optional): non-tradeable coins -> NaN signal, so rank() skips them.
    if tradeable is not None:
        signal = signal.where(tradeable)
    
    rank = signal.rank(axis=1)
    average_rank = rank.mean(axis=1)
    
    # portfolio
    deviance = rank.subtract(average_rank, axis=0)
    total = deviance.abs().sum(axis=1)
    weight = deviance.div(total,axis=0)
    # Non-tradeable coins -> weight 0. Done BEFORE the rebalance / ffill step so that
    # ffill cannot carry an old weight forward into an hour where the coin is excluded.
    if tradeable is not None:
        weight = weight.where(tradeable, 0)
    # Hold positions between rebalances.
    rebal = pd.Series(np.arange(len(weight)) % hold == 0, index=weight.index)
    weight = weight.where(rebal, axis=0).ffill()
    
    
    # returns
    # Weights formed at the close of hour t earn hour t+1's return (one-bar lag = no look-ahead).
    strat_ret = ret * weight.shift(1)
    port_ret = strat_ret.sum(axis=1)
    # Turnover lagged the same way as returns, so costs line up with the trades.
    turnover = weight.shift(1).subtract(weight.shift(2)).abs().sum(axis=1)
    tx_cost = turnover * bps
    net_ret = port_ret - tx_cost
    
       
    return pd.DataFrame({
        'gross':port_ret,
        'net':net_ret,
        'turnover':turnover
    })



In [13]:
# Baseline run: hor = 1 (last hour's return), rebalance every hour, 20 bps.
out = run_strategy(ret, hor=1, hold=1, bps=20/10000)
out

,gross,net,turnover
2020-01-01 00:00:00,0.0000,0.0000,0.0000
2020-01-01 01:00:00,0.0000,0.0000,0.0000
2020-01-01 02:00:00,-0.0005,-0.0005,0.0000
2020-01-01 03:00:00,0.0007,-0.0015,1.1000
2020-01-01 04:00:00,0.0015,-0.0017,1.6000
...,...,...,...
2026-08-30 20:00:00,0.0019,-0.0013,1.6000
2026-08-30 21:00:00,-0.0020,-0.0052,1.6000
2026-08-30 22:00:00,0.0001,-0.0021,1.1000
2026-08-30 23:00:00,-0.0004,-0.0028,1.2000


In [14]:
# ======================================================================
# 4. PERFORMANCE STATISTICS
# stats(): turns one hourly return series into summary numbers.
#  - Hourly returns are compounded into DAILY returns; Sharpe and vol are
#    annualized from daily data with sqrt(365) (crypto trades every day).
#  - Drawdown is measured on a compounded equity curve, (1 + r).cumprod().
#  - breakeven_bps = average return per unit of turnover, in bps: the cost
#    level at which the strategy breaks even. Only meaningful on the GROSS
#    series (on net it equals gross breakeven minus the assumed cost).
#  - BTC regression on daily returns: correlation, beta (= corr x vol ratio,
#    as in Homework 4), and an OLS alpha (annualized) with its t-stat.
# ======================================================================
def stats(r, turnover, btc):

    # r: one hourly return series (gross or net)
    # turnover: hourly turnover series
    # btc: hourly BTC returns

    # Compound hourly returns into daily returns.
    daily = (1 + r).resample('D').prod() - 1
    ann_vol = daily.std() * np.sqrt(365)
    daily_sharpe = daily.mean() / daily.std() * np.sqrt(365)
    # Drawdown from the running peak of the compounded equity curve.
    equity_curve = (1 + r).cumprod()
    drawdown = equity_curve.divide(equity_curve.cummax()) - 1
    max_drawdown = drawdown.min()
    breakeven_bps = r.mean() / turnover.mean() * 10000
    cum_return = (1 + r).prod() - 1
    # Benchmark: BTC daily returns, built from the btc input.
    btc_daily_ret = (1 + btc).resample('D').prod() - 1
    corr = daily.corr(btc_daily_ret)
    beta = corr * (daily.std() / btc_daily_ret.std())
    # Regress daily strategy returns on BTC: intercept = alpha (daily, x365 to annualize).
    model = sm.OLS(daily,sm.add_constant(btc_daily_ret)).fit()
    alpha = model.params.iloc[0] * 365
    alpha_tstat = model.tvalues.iloc[0]
    

    return {
    'daily_sharpe': daily_sharpe,
    'ann_vol': ann_vol,
    'cum_return': cum_return,
    'max_drawdown': max_drawdown,
    'turnover': turnover.mean(),
    'breakeven_bps': breakeven_bps,
    'beta': beta,
    'corr': corr,
    'alpha': alpha,
    'alpha_tstat': alpha_tstat,
    }

In [15]:
# Full-sample results (2020-2026), baseline hor = 1, hold = 1, 20 bps, gross vs net.
# A gross Sharpe near 19 is not achievable in practice: part of it is bid-ask
# bounce (closes alternate between bid and ask, which looks like reversal).
# Net is deeply negative: the edge per trade is smaller than the cost of trading.
pd.DataFrame({
    'gross': stats(out['gross'], out['turnover'],ret['BTCUSDT']),
    'net': stats(out['net'], out['turnover'],ret['BTCUSDT'])
    })

,gross,net
daily_sharpe,18.8810,-51.3407
ann_vol,0.3932,0.3372
cum_return,"1,203,359,332,422,255,378,432.0000",-1.0000
max_drawdown,-0.2993,-1.0000
turnover,1.4308,1.4308
breakeven_bps,5.8576,-14.1424
beta,-0.0586,-0.0511
corr,-0.0899,-0.0916
alpha,7.4566,-17.2842
alpha_tstat,49.1111,-132.7768


In [16]:
# ======================================================================
# 5. TRAIN / VALIDATION SPLIT
# First half of the hours = training (Jan 2020 - May 1 2023); second half =
# validation (May 2 2023 - Aug 2026). The strategy is run once on the full
# history and the OUTPUT is sliced, so the signal is already warm at the
# start of validation (it only looks backward, so nothing leaks).
# Parameters are chosen on training only; validation is looked at once.
# ======================================================================
# Split point: row n (iloc[:n] = train, iloc[n:] = validation, no overlap).
n = round(len(out)/2)

# Baseline (1, 1) at 20 bps: gross and net, train and validation.
strat_performance = pd.DataFrame({
    'Training_Gross':stats(
        out['gross'].iloc[:n], 
        out['turnover'].iloc[:n], 
        ret['BTCUSDT'].iloc[:n]),
    'Training_Net': stats(
        out['net'].iloc[:n], 
        out['turnover'].iloc[:n], 
        ret['BTCUSDT'].iloc[:n]),
    'Validation_Gross':stats(
        out['gross'].iloc[n:], 
        out['turnover'].iloc[n:], 
        ret['BTCUSDT'].iloc[n:]),
    'Validation_Net': stats(
        out['net'].iloc[n:], 
        out['turnover'].iloc[n:], 
        ret['BTCUSDT'].iloc[n:])
                         })

strat_performance

,Training_Gross,Training_Net,Validation_Gross,Validation_Net
daily_sharpe,15.5070,-48.4845,23.4978,-56.3786
ann_vol,0.4306,0.3715,0.3477,0.2947
cum_return,"2,838,906,449.8412",-1.0000,"423,881,291,355.2834",-1.0000
max_drawdown,-0.2993,-1.0000,-0.1314,-1.0000
turnover,1.4302,1.4302,1.4314,1.4314
breakeven_bps,5.2752,-14.7248,6.4395,-13.5605
beta,-0.0657,-0.0578,-0.0380,-0.0319
corr,-0.1102,-0.1124,-0.0499,-0.0494
alpha,6.7227,-17.9726,8.1863,-16.5997
alpha_tstat,28.6332,-88.7589,42.9947,-102.8780


In [17]:
# ======================================================================
# 6. PARAMETER AND COST SWEEP (TRAINING DATA ONLY)
# Every combination of lookback (hor) x holding period (hold) x cost.
# Small grid with interpretable values: 1 hour, 6 hours, 1 day, 3 days.
# Cost is an assumption, not a parameter: 7 bps (optimistic) vs 20 bps.
# Ranked by NET Sharpe (gross ignores turnover and would always favor the
# fastest, most expensive setting).
# Result: nothing is net-positive at 20 bps. At 7 bps only (1, 6) and (24, 72)
# are positive, exactly the two settings with breakeven above 7 bps.
# ======================================================================
hors = [1, 6, 24, 72]
holds = [1, 6, 24, 72]
costs = [7, 20]

# One entry per (hor, hold, cost) combination.
results = {}

for hor in hors:
    for hold in holds:
        for cost in costs:
            res = run_strategy(ret, hor = hor, hold = hold, bps = cost/10000)
            # Run on the full history, then score the training rows only.
            train = res.iloc[:n]
            s_gross = stats(train['gross'],train['turnover'],ret['BTCUSDT'].iloc[:n])
            s_net = stats(train['net'],train['turnover'],ret['BTCUSDT'].iloc[:n])
            results[(hor,hold, cost)]={
                'gross_sharpe':s_gross['daily_sharpe'],
                'net_sharpe':s_net['daily_sharpe'],
                'turnover':s_gross['turnover'],
                'breakeven_bps':s_gross['breakeven_bps']}

# Dict of dicts -> table; .T so each combination is a row (3-level index).
sweep = pd.DataFrame(results).T
sweep.index.names = ['hor', 'hold','cost']
sweep.sort_values('net_sharpe', ascending = False)

gross_sharpe  net_sharpe  turnover  breakeven_bps
hor hold cost                                                   
24  72   7           0.5687      0.2253    0.0185        11.6117
1   6    7           4.9580      0.2246    0.2197         7.4770
    24   7           0.9282     -0.0488    0.0553         6.7238
6   1    7           9.3655     -0.2639    0.6840         6.7961
24  72   20          0.5687     -0.4104    0.0185        11.6117
6   72   7          -0.1379     -0.4568    0.0179        -3.0532
1   72   7          -0.1429     -0.4863    0.0182        -2.7196
72  72   7          -0.4512     -0.8071    0.0185        -8.5667
24  24   7           0.0374     -0.9280    0.0562         0.3037
6   72   20         -0.1379     -1.0475    0.0179        -3.0532
    24   7          -0.1641     -1.0784    0.0548        -1.3160
1   72   20         -0.1429     -1.1246    0.0182        -2.7196
72  1    7           2.3946     -1.1719    0.2275         4.7135
    24   7          -0.6097     -1.1995    0.0333        -6.9734
24  1    7           4.3558     -1.4370    0.3824         5.3024
72  72   20         -0.4512     -1.4645    0.0185        -8.5667
    6    7          -0.5313     -1.7129    0.0735        -3.2571
1   24   20          0.9282     -1.8746    0.0553         6.7238
6   6    7           1.8259     -2.0715    0.2306         3.3090
24  6    7          -0.1782     -2.1590    0.1216        -0.6548
72  24   20         -0.6097     -2.2975    0.0333        -6.9734
24  24   20          0.0374     -2.7280    0.0562         0.3037
6   24   20         -0.1641     -2.7837    0.0548        -1.3160
72  6    20         -0.5313     -3.9358    0.0735        -3.2571
1   1    7          15.5070     -5.2665    1.4302         5.2752
24  6    20         -0.1782     -5.9350    0.1216        -0.6548
72  1    20          2.3946     -8.1144    0.2275         4.7135
1   6    20          4.9580     -8.7685    0.2197         7.4770
6   6    20          1.8259     -9.6100    0.2306         3.3090
24  1    20          4.3558    -13.2416    0.3824         5.3024
6   1    20          9.3655    -20.0782    0.6840         6.7961
1   1    20         15.5070    -48.4845    1.4302         5.2752

In [18]:
# ======================================================================
# 7. VALIDATION OF PRE-COMMITTED PICKS
# Settings chosen from training BEFORE looking at validation:
#   (1, 1)   baseline
#   (1, 6)   from the smooth, robust region of the training grid
#   (24, 72) best training net Sharpe, but an isolated bump in the grid
# Each is run once and scored on validation rows only.
# Result: (24, 72) fails out of sample (likely noise); (1, 6) stays positive
# at 7 bps. Nothing survives 20 bps. (Section 9 shows that the (1, 6) result
# was largely driven by stale prices.)
# ======================================================================
picks = [(1, 1), (1, 6), (24, 72)]
val_results = {}

# Each item in picks is a (hor, hold) pair, unpacked into two names.
for hor, hold in picks:
    for cost in costs:
        res = run_strategy(ret, hor = hor, hold = hold, bps = cost/10000)
        val = res.iloc[n:]
        s_gross = stats(val['gross'],val['turnover'],ret['BTCUSDT'].iloc[n:])
        s_net = stats(val['net'],val['turnover'],ret['BTCUSDT'].iloc[n:])
        val_results[(hor,hold, cost)]={
            'gross_sharpe':s_gross['daily_sharpe'],
            'net_sharpe':s_net['daily_sharpe'],
            'turnover':s_gross['turnover'],
            'breakeven_bps':s_gross['breakeven_bps']}

val_sweep = pd.DataFrame(val_results).T
val_sweep.index.names = ['hor', 'hold','cost']
val_sweep.sort_values('net_sharpe', ascending = False)

gross_sharpe  net_sharpe  turnover  breakeven_bps
hor hold cost                                                   
1   6    7           7.4561      1.7106    0.2179         9.1294
24  72   7           0.2436     -0.2759    0.0186         3.9189
         20          0.2436     -1.2335    0.0186         3.9189
1   1    7          23.4978     -2.1095    1.4314         6.4395
    6    20          7.4561     -9.2511    0.2179         9.1294
    1    20         23.4978    -56.3786    1.4314         6.4395

In [19]:
# ======================================================================
# 8. VOLUME INTERACTION
# Hypothesis: moves on unusually LOW volume are more likely liquidity noise
# and should reverse more; moves on HIGH volume may carry information and
# reverse less.
# ======================================================================
# Relative volume = this hour's volume / the coin's average volume over the
# PREVIOUS 24 hours (shift(1) keeps the current hour out of its own baseline).
# Raw volume would just rank coins by size (BTC >> XLM).
# >1 = busier than normal, <1 = quieter than normal.
relvol = vol / vol.shift(1).rolling(24, min_periods = 24).mean()
relvol

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,1.0757,1.6779,0.3999,0.8010,4.1992,1.7965,0.8324,1.5741,1.7936
2026-08-30 21:00:00,1.0128,3.0262,0.4803,0.9354,0.0327,0.8232,0.3842,1.5142,2.1382
2026-08-30 22:00:00,0.7893,1.8884,2.8127,0.3972,0.1519,12.0134,1.3849,1.3006,4.9948
2026-08-30 23:00:00,5.1368,7.0782,2.1876,4.0171,1.9792,1.2676,6.0183,0.6383,3.6512


In [20]:
# Liquidity filter (rule fixed in advance, uses past data only).
# Finding: many coins had hours with ZERO trades on Binance.US (e.g. ETC in about
# 25% of all hours, BCH about 14%), especially from 2022 on. In a dead hour the close
# is a stale price; the next trade then jumps, which looks like a reversal that
# could not actually be traded.
# Count hours with no trades (zero OR missing volume) in the last 24 hours,
# including the current hour. ~(vol > 0) is True for 0 and for NaN.
dead_hours = (~(vol > 0)).rolling(24, min_periods = 24).sum()
# A coin is ok this hour only if it traded in every one of the last 24 hours.
coin_ok = dead_hours == 0 
# Need at least 4 ok coins in the hour (at least two longs and two shorts).
enough = coin_ok.sum(axis = 1) >= 4

In [21]:
# tradeable[hour, coin] = coin is ok AND its hour has enough ok coins.
# coin_ok is a grid and enough is one value per hour; axis=0 applies each
# hour's value across all coins. Result: True/False, hours x coins.
# About 7% of hours have no tradeable coins (the strategy is flat then).
tradeable = coin_ok.mul(enough, axis = 0)
tradeable

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,False,False,False,False,False,False,False,False,False
2020-01-01 01:00:00,False,False,False,False,False,False,False,False,False
2020-01-01 02:00:00,False,False,False,False,False,False,False,False,False
2020-01-01 03:00:00,False,False,False,False,False,False,False,False,False
2020-01-01 04:00:00,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,True,True,False,True,False,False,False,False,True
2026-08-30 21:00:00,True,True,False,True,False,False,False,False,True
2026-08-30 22:00:00,True,True,False,True,False,False,False,False,True
2026-08-30 23:00:00,True,True,False,True,False,False,False,False,True


In [22]:
# Volume percentile of each tradeable coin within its hour: 0 = quietest,
# 1 = busiest. Percentiles (pct=True) are comparable across hours with
# different numbers of tradeable coins. Non-tradeable coins = NaN.
vol_rank = (relvol * tradeable).where(tradeable).rank(axis = 1, pct=True)
vol_rank

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,0.5000,0.7500,NaN,0.2500,NaN,NaN,NaN,NaN,1.0000
2026-08-30 21:00:00,0.5000,1.0000,NaN,0.2500,NaN,NaN,NaN,NaN,0.7500
2026-08-30 22:00:00,0.5000,0.7500,NaN,0.2500,NaN,NaN,NaN,NaN,1.0000
2026-08-30 23:00:00,0.7500,1.0000,NaN,0.5000,NaN,NaN,NaN,NaN,0.2500


In [23]:
# Hypothesis test, step 1: relative returns THIS hour.
# Keep tradeable coins only (.where -> NaN elsewhere), then subtract the hour's
# average across those coins. The strategy is long-short, so only moves
# RELATIVE to the other coins matter; market-wide moves cancel out.
ret_t = ret.where(tradeable)
ret_t

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,-0.0032,-0.0043,NaN,-0.0021,NaN,NaN,NaN,NaN,-0.0053
2026-08-30 21:00:00,0.0003,-0.0038,NaN,-0.0022,NaN,NaN,NaN,NaN,-0.0073
2026-08-30 22:00:00,-0.0025,-0.0043,NaN,-0.0041,NaN,NaN,NaN,NaN,-0.0119
2026-08-30 23:00:00,-0.0095,-0.0216,NaN,-0.0120,NaN,NaN,NaN,NaN,-0.0164


In [24]:
# Average return of the tradeable coins in each hour (NaN skipped).
avg_ret_t = ret_t.mean(axis=1)
avg_ret_t

2020-01-01 00:00:00       NaN
2020-01-01 01:00:00       NaN
2020-01-01 02:00:00       NaN
2020-01-01 03:00:00       NaN
2020-01-01 04:00:00       NaN
                        ...  
2026-08-30 20:00:00   -0.0037
2026-08-30 21:00:00   -0.0032
2026-08-30 22:00:00   -0.0057
2026-08-30 23:00:00   -0.0149
2026-08-31 00:00:00    0.0032
Freq: h, Length: 58417, dtype: float64

In [25]:
# Each coin's return relative to the hour's average (each row sums to about 0).
rel_ret_t = ret_t.sub(avg_ret_t, axis = 0)
rel_ret_t

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,0.0005,-0.0006,NaN,0.0016,NaN,NaN,NaN,NaN,-0.0015
2026-08-30 21:00:00,0.0036,-0.0006,NaN,0.0010,NaN,NaN,NaN,NaN,-0.0041
2026-08-30 22:00:00,0.0032,0.0014,NaN,0.0016,NaN,NaN,NaN,NaN,-0.0063
2026-08-30 23:00:00,0.0054,-0.0067,NaN,0.0029,NaN,NaN,NaN,NaN,-0.0015


In [26]:
# Step 2: relative returns NEXT hour, for the same coins.
# shift(-1) moves next hour's return up into this hour's row; the mask is this
# hour's tradeable set, so both grids cover exactly the same coins.
# (Using the future is fine here: this measures the hypothesis, it is not a trading rule.)
ret_n = ret.shift(-1).where(tradeable)
ret_n

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,0.0003,-0.0038,NaN,-0.0022,NaN,NaN,NaN,NaN,-0.0073
2026-08-30 21:00:00,-0.0025,-0.0043,NaN,-0.0041,NaN,NaN,NaN,NaN,-0.0119
2026-08-30 22:00:00,-0.0095,-0.0216,NaN,-0.0120,NaN,NaN,NaN,NaN,-0.0164
2026-08-30 23:00:00,0.0030,0.0041,NaN,0.0020,NaN,NaN,NaN,NaN,0.0037


In [27]:
# Average next-hour return of the same coins.
avg_ret_n = ret_n.mean(axis=1)
avg_ret_n

2020-01-01 00:00:00       NaN
2020-01-01 01:00:00       NaN
2020-01-01 02:00:00       NaN
2020-01-01 03:00:00       NaN
2020-01-01 04:00:00       NaN
                        ...  
2026-08-30 20:00:00   -0.0032
2026-08-30 21:00:00   -0.0057
2026-08-30 22:00:00   -0.0149
2026-08-30 23:00:00    0.0032
2026-08-31 00:00:00       NaN
Freq: h, Length: 58417, dtype: float64

In [28]:
# Next-hour return relative to that average.
rel_ret_n = ret_n.sub(avg_ret_n, axis = 0)
rel_ret_n

,BTCUSDT,ETHUSDT,ADAUSDT,BNBUSDT,LTCUSDT,BCHUSDT,XLMUSDT,ETCUSDT,DOGEUSDT
2020-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,0.0036,-0.0006,NaN,0.0010,NaN,NaN,NaN,NaN,-0.0041
2026-08-30 21:00:00,0.0032,0.0014,NaN,0.0016,NaN,NaN,NaN,NaN,-0.0063
2026-08-30 22:00:00,0.0054,-0.0067,NaN,0.0029,NaN,NaN,NaN,NaN,-0.0015
2026-08-30 23:00:00,-0.0002,0.0009,NaN,-0.0012,NaN,NaN,NaN,NaN,0.0004


In [29]:
# Step 3: one long table, one row per tradeable coin-hour.
# stack() turns each hours x coins grid into a long column labeled (hour, coin);
# the dict makes three named columns; dropna() keeps only coin-hours with all
# three values (drops the final hour and hours just before data gaps).
stacked_rel_ret_t = rel_ret_t.stack()
stacked_rel_ret_n = rel_ret_n.stack()
stacked_vol_rank = vol_rank.stack()
coin_hours = {
    'rel_ret': stacked_rel_ret_t,
    'rel_ret_next': stacked_rel_ret_n,
    'vol_rank': stacked_vol_rank,
}
coin_hours = pd.DataFrame(coin_hours).dropna()
coin_hours

rel_ret  rel_ret_next  vol_rank
2020-01-02 00:00:00 ETHUSDT   -0.0012        0.0023    0.1667
                    ADAUSDT    0.0003       -0.0006    0.5000
                    BNBUSDT    0.0003        0.0001    1.0000
                    LTCUSDT    0.0024        0.0027    0.6667
                    XLMUSDT   -0.0010        0.0005    0.8333
...                               ...           ...       ...
2026-08-30 22:00:00 DOGEUSDT  -0.0063       -0.0015    1.0000
2026-08-30 23:00:00 BTCUSDT    0.0054       -0.0002    0.7500
                    ETHUSDT   -0.0067        0.0009    1.0000
                    BNBUSDT    0.0029       -0.0012    0.5000
                    DOGEUSDT  -0.0015        0.0004    0.2500

[366327 rows x 3 columns]

In [30]:
# Step 4: training period only (decisions are made on training data).
# The long table has several rows per hour, so iloc[:n] would be wrong here;
# instead convert row n of the hourly index into a date and filter by date.
cutoff = out.index[n]
hours = coin_hours.index.get_level_values(0)
coin_hours_train = coin_hours[hours < cutoff]
coin_hours_train

rel_ret  rel_ret_next  vol_rank
2020-01-02 00:00:00 ETHUSDT   -0.0012        0.0023    0.1667
                    ADAUSDT    0.0003       -0.0006    0.5000
                    BNBUSDT    0.0003        0.0001    1.0000
                    LTCUSDT    0.0024        0.0027    0.6667
                    XLMUSDT   -0.0010        0.0005    0.8333
...                               ...           ...       ...
2023-05-01 23:00:00 ETHUSDT    0.0006        0.0013    0.6667
                    ADAUSDT    0.0001        0.0011    0.3333
                    BNBUSDT   -0.0016       -0.0042    1.0000
                    LTCUSDT   -0.0001        0.0008    0.1667
                    DOGEUSDT   0.0011        0.0017    0.5000

[195103 rows x 3 columns]

In [31]:
# Step 5: split into QUIET coin-hours (bottom half of volume within the hour) ...
# Build the True/False mask, then keep the matching rows (all columns).
coin_hours_quiet = coin_hours_train['vol_rank'] <= 0.5
coin_hours_low_vol = coin_hours_train[coin_hours_quiet]
coin_hours_low_vol

rel_ret  rel_ret_next  vol_rank
2020-01-02 00:00:00 ETHUSDT   -0.0012        0.0023    0.1667
                    ADAUSDT    0.0003       -0.0006    0.5000
                    DOGEUSDT  -0.0007       -0.0049    0.3333
2020-01-02 01:00:00 ETHUSDT    0.0028       -0.0020    0.2000
                    ADAUSDT   -0.0001       -0.0001    0.4000
...                               ...           ...       ...
2023-05-01 22:00:00 BNBUSDT   -0.0018       -0.0016    0.3333
                    DOGEUSDT  -0.0011        0.0011    0.5000
2023-05-01 23:00:00 ADAUSDT    0.0001        0.0011    0.3333
                    LTCUSDT   -0.0001        0.0008    0.1667
                    DOGEUSDT   0.0011        0.0017    0.5000

[88990 rows x 3 columns]

In [32]:
# ... and BUSY coin-hours (top half). Busy has slightly more rows: with an odd
# number of coins, the middle coin's percentile lands above 0.5.
coin_hours_busy = coin_hours_train['vol_rank'] > 0.5
coin_hours_high_vol = coin_hours_train[coin_hours_busy]
coin_hours_high_vol

rel_ret  rel_ret_next  vol_rank
2020-01-02 00:00:00 BNBUSDT   0.0003        0.0001    1.0000
                    LTCUSDT   0.0024        0.0027    0.6667
                    XLMUSDT  -0.0010        0.0005    0.8333
2020-01-02 01:00:00 BNBUSDT   0.0007       -0.0018    0.6000
                    XLMUSDT   0.0010       -0.0002    0.8000
...                              ...           ...       ...
2023-05-01 22:00:00 ETHUSDT  -0.0010        0.0006    0.8333
                    LTCUSDT   0.0025       -0.0001    0.6667
2023-05-01 23:00:00 BTCUSDT  -0.0001       -0.0008    0.8333
                    ETHUSDT   0.0006        0.0013    0.6667
                    BNBUSDT  -0.0016       -0.0042    1.0000

[106113 rows x 3 columns]

In [33]:
# Step 6: reversal strength = correlation between this hour's and next hour's
# relative return. Negative = moves tend to reverse.
# Result (train): quiet = -0.088.
coin_hours_low_vol_corr = coin_hours_low_vol['rel_ret'].corr(coin_hours_low_vol['rel_ret_next'])
coin_hours_low_vol_corr

np.float64(-0.08768808259805685)

In [34]:
# Same for busy coin-hours. Result (train): busy = -0.067.
# Quiet moves reverse about 30% more strongly, as the hypothesis predicts.
coin_hours_high_vol_corr = coin_hours_high_vol['rel_ret'].corr(coin_hours_high_vol['rel_ret_next'])
coin_hours_high_vol_corr

np.float64(-0.06662658257943299)

In [35]:
# Step 7: robustness - is the quiet vs busy gap stable year by year?
# For each year: the correlation and row count in each group, stored under the
# year key; .T makes one row per year.
# Result: quiet reverses more in 2020, 2022 and (weakly) 2023; no difference in
# 2021, the most liquid year. Rough noise per correlation is about 1/sqrt(n), so
# the 2020 and 2022 gaps are clear and 2023 is marginal.
# Caveat: quiet hours have fewer trades and so more bid-ask bounce; part of the
# quiet-volume reversal may not be tradeable.
years = coin_hours_low_vol.index.get_level_values(0).year.unique()
year_corr = {}

for year in years:
    # Quiet group, this year.
    rel_ret_year_low = coin_hours_low_vol[coin_hours_low_vol.index.get_level_values(0).year == year]['rel_ret']
    rel_ret_next_year_low = coin_hours_low_vol[coin_hours_low_vol.index.get_level_values(0).year == year]['rel_ret_next']
    quiet_corr = rel_ret_year_low.corr(rel_ret_next_year_low)
    low_vol_n = len(rel_ret_year_low)
    
    # Busy group, this year.
    rel_ret_year_high = coin_hours_high_vol[coin_hours_high_vol.index.get_level_values(0).year == year]['rel_ret']
    rel_ret_next_year_high = coin_hours_high_vol[coin_hours_high_vol.index.get_level_values(0).year == year]['rel_ret_next']
    busy_corr = rel_ret_year_high.corr(rel_ret_next_year_high)
    high_vol_n = len(rel_ret_year_high)
    
    # Add one entry per year (year_corr[year] = ... adds; year_corr = ... would overwrite).
    year_corr [year]= {
        'quiet_corr': quiet_corr,
        'quiet_n':low_vol_n,
        'busy_corr': busy_corr,
        'busy_n': high_vol_n,
    }
pd.DataFrame(year_corr).T

,quiet_corr,quiet_n,busy_corr,busy_n
2020,-0.1857,"17,045.0000",-0.1332,"20,026.0000"
2021,-0.0485,"34,658.0000",-0.0545,"42,700.0000"
2022,-0.1046,"29,163.0000",-0.0618,"33,895.0000"
2023,-0.0979,"8,124.0000",-0.0809,"9,492.0000"


In [44]:
# ======================================================================
# 9. DOES THE LIQUIDITY FILTER CHANGE THE RESULTS?
# Same strategy with and without the tradeable mask, for the two key settings,
# on training and validation, at 7 bps.
# Question: how much of the original edge came from stale prices?
# ======================================================================
# Result: gross Sharpe falls everywhere with the filter, and much more in
# validation, where exchange liquidity collapsed: (1, 1) -28% train vs -41% val;
# (1, 6) -39% vs -65%. Breakeven drops below 7 bps in every filtered case, and
# (1, 6)'s positive validation result (net +1.71) turns negative (-3.15).
# Caveat: the filter also shrinks the universe, so part of the drop comes from
# fewer bets, not only from removing the stale-price artifact.
settings = [(1, 1), (1, 6)]
# Each version = (label, mask). None = no filter, i.e. the original strategy.
versions = {'original': None, 'filtered': tradeable}
# BTC benchmark sliced to each period once (it never changes inside the loop).
btc_train = ret['BTCUSDT'].iloc[:n]          
btc_val   = ret['BTCUSDT'].iloc[n:]

comparison = {}

# 2 settings x 2 versions = 4 runs; each run is split into train and val -> 8 rows.
for hor, hold in settings:                                 
    for version, mask in versions.items():                 

        
        # Run once on the full history, then slice into the two periods.
        edge = run_strategy(ret, hor=hor, hold=hold, bps=7/10000, tradeable=mask)

        train = edge.iloc[:n]
        val   = edge.iloc[n:]

        # Gross and net stats for each period (BTC sliced to the same period).
        train_gross = stats(train['gross'], train['turnover'], btc_train)
        train_net   = stats(train['net'],   train['turnover'], btc_train)
        val_gross   = stats(val['gross'],   val['turnover'],   btc_val)
        val_net     = stats(val['net'],     val['turnover'],   btc_val)

       
        # One row per period; breakeven comes from the GROSS stats.
        comparison[(hor, hold, version, 'train')] = {
            'gross_sharpe':  train_gross['daily_sharpe'],
            'net_sharpe':    train_net['daily_sharpe'],
            'turnover':      train_gross['turnover'],
            'breakeven_bps': train_gross['breakeven_bps'],
        }
        comparison[(hor, hold, version, 'val')] = {
            'gross_sharpe':  val_gross['daily_sharpe'],
            'net_sharpe':    val_net['daily_sharpe'],
            'turnover':      val_gross['turnover'],
            'breakeven_bps': val_gross['breakeven_bps'],
        }

# Table with a 4-level index: hor, hold, version, period.
comparison = pd.DataFrame(comparison).T
comparison.index.names = ['hor', 'hold', 'version', 'period']
comparison

gross_sharpe  net_sharpe  turnover  breakeven_bps
hor hold version  period                                                   
1   1    original train        15.5070     -5.2665    1.4302         5.2752
                  val          23.4978     -2.1095    1.4314         6.4395
         filtered train        11.1579     -9.2971    1.2905         3.9061
                  val          13.8385    -17.9871    1.3094         3.1703
    6    original train         4.9580      0.2246    0.2197         7.4770
                  val           7.4561      1.7106    0.2179         9.1294
         filtered train         3.0021     -1.1765    0.2044         5.1816
                  val           2.6012     -3.1468    0.2065         3.2373